# 00 — Setup

Checks that the environment is complete and that every checkpoint and dataset the
paper notebooks need is present and matches its sha256 in `checkpoints/MANIFEST.json`.

Large artifacts are not stored in git. `scripts/manifest.py verify --fetch-missing`
copies anything missing from the source recorded in the manifest (currently the
authors' cluster backup; to be replaced by a public download URL).

| Notebook | Paper items |
|---|---|
| `01_toy_multimodal` | Fig 1, Table 12 |
| `02_dna_base_model` | Tables 7–11, 16; Fig 6 |
| `03_dna_cyclizability` | Table 1, Figs 3, 7, 8; Tables 4–5 |
| `04_dna_exact_rewards` | Table 13 |
| `05_dna_mc_scaling` | Fig 5; Tables 14, 15, 17, 18 |
| `06_dna_benchmarks` | Fig 4 |
| `07_qm9_property_guidance` | Tables 2, 19–22 (QM9) |
| `08_geom_fingerprint` | Table 3; GEOM rows of Tables 19–21 |
| `09_evo2_plausibility` | Table 6 |

In [1]:
FETCH_MISSING = False  # set True to copy missing artifacts from their recorded source

from _common import *
import importlib
import subprocess
import sys

## Environment

In [2]:
mods = ["torch", "lightning", "pytorch_lightning", "tensordict", "rdkit", "posebusters",
        "sklearn", "matplotlib", "tabasco", "multimfm", "dmfm", "toy"]
rows = []
for m in mods:
    try:
        mod = importlib.import_module(m)
        rows.append({"module": m, "version": getattr(mod, "__version__", ""), "ok": True})
    except Exception as e:  # report, don't stop: some packages may still be in progress
        rows.append({"module": m, "version": f"{type(e).__name__}: {e}", "ok": False})
print(sys.version)
pd.DataFrame(rows)

/n/home04/uunneberg/micromamba/envs/multimfm/lib/python3.11/site-packages/torchmetrics/utilities/imports.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


3.11.15 | packaged by conda-forge | (main, Jun 11 2026, 03:34:02) [GCC 14.3.0]


,module,version,ok
0,torch,2.5.1+cu124,True
1,lightning,2.6.5,True
2,pytorch_lightning,2.6.5,True
3,tensordict,0.6.2,True
4,rdkit,2024.09.4,True
5,posebusters,0.3.1,True
6,sklearn,1.9.0,True
7,matplotlib,3.10.9,True
8,tabasco,,True
9,multimfm,,True


In [3]:
import torch
print("CUDA available:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu (RERUN=True cells need a GPU)")

CUDA available: False | device: cpu (RERUN=True cells need a GPU)


## Artifacts

In [4]:
cmd = [sys.executable, str(REPO / "scripts" / "manifest.py"), "verify"]
if FETCH_MISSING:
    cmd.append("--fetch-missing")
out = subprocess.run(cmd, capture_output=True, text=True)
lines = out.stdout.strip().splitlines()
print(lines[-1])
problems = [l for l in lines if l.startswith(("MISSING", "MISMATCH"))]
print("\n".join(problems) if problems else "")

all artifacts verified



In [5]:
manifest = json.loads((CHECKPOINTS / "MANIFEST.json").read_text())["artifacts"]
df = pd.DataFrame([{"path": p, "group": e.get("group", ""), "MB": round(e["size"] / 1e6, 1),
                    "note": e.get("note", "")} for p, e in manifest.items()])
df.groupby("group")["MB"].agg(["count", "sum"]).round(1)

,count,sum
group,,
dna,41,884.3
geom,11,595.4
qm9,4,47.1
toy,3,46.3
